# Convolutional Neural Network (CNN) with PyTorch

### SVL Technologies — Practical Deep Learning Notebook

This notebook explains CNN concepts step by step and builds a complete image-classification model using PyTorch.

**Learning objectives**
- Understand convolution, filters, feature maps, stride and padding
- Understand ReLU and pooling
- Build and train a CNN in PyTorch
- Plot loss and accuracy
- Evaluate the model and make predictions
- Understand common CNN mistakes and improvement techniques

## 1. What is a CNN?

A **Convolutional Neural Network (CNN)** is a deep learning architecture designed especially for images and other spatial data.

A CNN learns a hierarchy of features:

**Pixels → Edges → Textures/Shapes → Object features → Class prediction**

Typical flow:

**Image → Convolution → ReLU → Pooling → Convolution → ReLU → Pooling → Flatten → Fully Connected → Output**

CNNs are effective because they preserve spatial structure, use shared filters, and learn visual features automatically.

## 2. Why CNN instead of a normal ANN?

If a 28×28 image is flattened, it becomes 784 independent values. A fully connected ANN does not naturally understand that nearby pixels form meaningful patterns.

CNNs use:

1. **Local connectivity** — examine small image regions
2. **Parameter sharing** — reuse the same filter across the image
3. **Feature maps** — preserve spatial information
4. **Hierarchical learning** — combine simple features into complex ones

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

print("PyTorch version:", torch.__version__)

## 3. Select CPU or GPU

PyTorch can use a CUDA-enabled GPU when available. Otherwise, the notebook runs on the CPU.

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## 4. Image tensors in PyTorch

PyTorch normally uses the format:

**N × C × H × W**

- N = batch size
- C = channels
- H = height
- W = width

A grayscale 28×28 image has shape **1 × 28 × 28**.

A batch of 64 images has shape **64 × 1 × 28 × 28**.

## 5. Convolution

A **kernel/filter** slides over an image and performs element-wise multiplication followed by summation.

The filter weights are learned during training.

Important parameters:

- **Kernel size:** e.g. 3×3
- **Stride:** how far the filter moves
- **Padding:** border added around the input
- **Number of filters:** number of output feature maps

Output-size formula:

**Output = floor((N + 2P − K) / S) + 1**

For N=28, K=3, P=1, S=1:

**Output = 28**

## 6. ReLU activation

ReLU means **Rectified Linear Unit**.

**ReLU(x) = max(0, x)**

Examples:

- ReLU(-3) = 0
- ReLU(2) = 2

ReLU introduces non-linearity, allowing the network to learn complex patterns.

## 7. Pooling

Pooling reduces spatial dimensions.

**MaxPool2d(2)** selects the maximum value from each 2×2 region.

For example:

```text
1  5
2  3
```

Max = **5**

A 2×2 pooling operation with stride 2 changes approximately:

**28×28 → 14×14**

Pooling reduces computation and creates more compact feature representations.

## 8. Dataset — MNIST

We will use **MNIST**, a handwritten-digit dataset.

Each image:
- is grayscale
- is 28×28 pixels
- belongs to one of 10 classes: 0–9

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=64, shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=64, shuffle=False
)

print("Training images:", len(train_dataset))
print("Testing images :", len(test_dataset))

## 9. Visualize the data

Always inspect your dataset before training a model.

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(10,4))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"Label: {labels[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 10. Build the CNN

Our model contains two convolution blocks:

**Conv2D → ReLU → MaxPool**

Then:

**Flatten → Linear → ReLU → Linear**

Shape calculation:

- Input: **1×28×28**
- After Pool 1: **16×14×14**
- After Pool 2: **32×7×7**
- Flatten: **32×7×7 = 1568**
- Classifier: **1568 → 128 → 10**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN().to(device)
print(model)

## 11. Count trainable parameters

Every convolution filter and fully connected weight is learned during training.

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", total_params)

## 12. Loss function and optimizer

This is a **10-class classification** problem.

We use:

- **CrossEntropyLoss** — measures classification error
- **Adam** — updates the model parameters
- Learning rate = **0.001**

Important: when using `CrossEntropyLoss`, give it the raw logits. Do **not** apply Softmax before the loss.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(criterion)
print(optimizer)

## 13. Training the CNN

Training steps:

1. Get a batch
2. Move data to the device
3. Clear old gradients
4. Forward pass
5. Calculate loss
6. Backpropagation
7. Update weights
8. Repeat

In [ ]:
num_epochs = 5

train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)   
    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

## 14. Plot training loss

A decreasing loss generally indicates that the model is learning to reduce prediction error.

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, marker="o")
plt.title("CNN Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

## 15. Plot training accuracy

Accuracy is the percentage of correctly classified training images.

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_accuracies, marker="o")
plt.title("CNN Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.grid(True)
plt.show()

## 16. Evaluate on unseen test data

We use:

- `model.eval()` — evaluation mode
- `torch.no_grad()` — disables gradient calculation

This gives a more realistic estimate of generalization.

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

## 17. Make predictions

The final layer produces 10 logits. The class with the largest logit is selected as the prediction.

In [ ]:
images, labels = next(iter(test_loader))
images_device = images.to(device)

model.eval()
with torch.no_grad():
    outputs = model(images_device)
    predictions = torch.argmax(outputs, dim=1)

plt.figure(figsize=(12,6))
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    actual = labels[i].item()
    predicted = predictions[i].item()
    plt.title(f"Actual: {actual} | Pred: {predicted}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 18. Confusion matrix

A confusion matrix shows which classes are being confused.

For example, a model may sometimes confuse visually similar digits such as 4 and 9 or 3 and 5.

In [ ]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

conf_matrix = np.zeros((10, 10), dtype=int)

for actual, predicted in zip(all_labels, all_predictions):
    conf_matrix[actual, predicted] += 1

plt.figure(figsize=(8,7))
plt.imshow(conf_matrix)
plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(range(10))
plt.yticks(range(10))
plt.colorbar()
plt.show()

## 19. Save the trained model

Saving `state_dict()` is the recommended common PyTorch approach for storing learned parameters.

In [ ]:
model_path = "simple_cnn_mnist.pth"
torch.save(model.state_dict(), model_path)
print("Saved:", model_path)

## 20. Load the model

The same architecture must be created before loading its parameters.

In [ ]:
loaded_model = SimpleCNN().to(device)
loaded_model.load_state_dict(torch.load(model_path, map_location=device))
loaded_model.eval()

print("Model loaded successfully.")

## 21. Common CNN mistakes

### 1. Wrong tensor shape
PyTorch expects **N×C×H×W**.

### 2. Wrong Linear size
Calculate the feature-map dimensions after every convolution and pooling layer.

### 3. Softmax before CrossEntropyLoss
Use raw logits directly with `CrossEntropyLoss`.

### 4. Forgetting model modes
Use `model.train()` during training and `model.eval()` during evaluation.

### 5. Evaluating only training accuracy
Always use unseen validation/test data.

### 6. Overfitting
If training accuracy keeps increasing while validation/test performance stops improving, consider augmentation, dropout, weight decay, early stopping or transfer learning.

## 22. How to improve this CNN

Try:

- More convolution layers
- More filters
- Batch Normalization
- Dropout
- Data augmentation
- Learning-rate scheduling
- SGD instead of Adam
- Transfer learning
- ResNet or other modern architectures

A larger model is not automatically better. Compare training and validation/test performance.

## 23. Student exercises

**Exercise 1:** Change 16 filters to 32 in the first convolution and recalculate parameters.

**Exercise 2:** Replace MaxPool2d with AveragePool2d.

**Exercise 3:** Add a third convolution block.

**Exercise 4:** Add Dropout and compare results.

**Exercise 5:** Train using SGD and compare it with Adam.

**Exercise 6:** Try learning rates `0.1`, `0.01`, `0.001`, and `0.0001`.

**Exercise 7:** Build a CNN for your own image dataset, such as:
- Cat vs Dog
- Plant Disease
- Fruit Classification
- Hand Gesture Recognition
- Vehicle Classification

# 24. Quick Revision

| Concept | Meaning |
|---|---|
| CNN | Neural network specialized for spatial data |
| Filter/Kernel | Learnable feature detector |
| Convolution | Applies filters to local image regions |
| Feature Map | Output produced by a filter |
| Stride | Filter movement step |
| Padding | Border added around input |
| ReLU | Non-linear activation |
| Pooling | Reduces spatial dimensions |
| Flatten | Converts feature maps into a vector |
| Loss | Measures prediction error |
| Optimizer | Updates model parameters |
| Epoch | One pass through the training dataset |

### Final CNN flow

**Image → Convolution → ReLU → Pooling → Feature Extraction → Classification**

### SVL Technologies
**Trusted Training Partner**

*Empower Your Future with SVL Technologies*